# Week 1 Day 5 Lab 1 - Harness Engineering

## The Claude Agent SDK works the queue on its own

The Claude Agent SDK is the harness behind Claude Code, packaged as a Python library. It brings the agent loop, tools to read and write files, a task list and subagents, all ready made.

We give it a folder with five claims from the queue, the policy wording and the claims manual, and ask it to decide every claim. We don't write any tools. Then we check its decisions against the handlers' answers.

## 4 parts:

PART 1: The claims desk

PART 2: Setting up the harness

PART 3: Working the queue

PART 4: How it did

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab uses Claude Sonnet 5.5 through Anthropic's API, and a run on five claims costs about 40 cents.
            You'll need an Anthropic API key with some credit, from <a href="https://console.anthropic.com">console.anthropic.com</a>, as <code>ANTHROPIC_API_KEY</code> in your <code>.env</code> file.<br/><br/>
            If you'd rather not spend anything, please watch this one. Lab 2 in <a href="week1day5_lab2.ipynb">week1day5_lab2.ipynb</a> does the same thing with Deep Agents on OpenRouter, and it has a free option.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
import time
from pathlib import Path

import pandas as pd
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from IPython.display import Markdown, display
from langchain.agents.middleware import ModelRetryMiddleware, TodoListMiddleware
from langchain_openai import ChatOpenAI

# Substituted: the Claude Agent SDK needs an Anthropic API key and Claude Code CLI,
# neither of which is installed here. We use Deep Agents on a local Ollama model instead.

In [2]:
# No Anthropic key is used: this run uses the local qwen3.5:4b model through Ollama.

## Part 1: The claims desk

The seed data is in the repo's `data` folder, as on the other days. We use the five claims marked for the tiny version of the queue, which include one that needs the claims manual.

In [3]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [4]:
claims, labels = get_sample_data(tiny=True)

for claim in claims:
    print(claim["claim_id"], claim["summary"])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


The agent works in a folder of its own, which we'll call the claims desk. It gets one file for each claim in the queue, a file for every policy and every customer, and the policy wording, the claims manual and the company's limits.

It never sees the handlers' decisions, which stay in `data/evals`. We keep those to score the agent at the end.

In [5]:
DESK = Path("lab1_desk")


def make_desk(claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (DESK / folder).mkdir(parents=True)
    for claim in claims:
        (DESK / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (DESK / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (DESK / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, DESK / name)


make_desk(claims)
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

In [6]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a to-do list with one item for each claim, and give each claim to the claim-handler agent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">The options</h2>
            <span style="color:#118AB2;"><code>tools</code> picks the built-in tools the agent can use: reading, searching and writing files, the task list (<code>TaskCreate</code>, <code>TaskUpdate</code>, <code>TaskList</code>, <code>TaskGet</code>) and <code>Task</code>, which hands work to a subagent. We leave out the shell and the web.<br/><br/>
            <code>agents</code> defines our subagent. <code>system_prompt</code> starts from Claude Code's own system prompt and adds our instructions.<br/><br/>
            <code>setting_sources=[]</code> keeps your own Claude Code settings out of the run. In <code>env</code>, <code>CLAUDE_CODE_DISABLE_AUTO_MEMORY</code> turns off the notes Claude Code keeps between sessions, so every run starts fresh, and <code>ENABLE_CLAUDEAI_MCP_SERVERS</code> leaves out any connectors from your claude.ai account.<br/><br/>
            <code>max_budget_usd</code> stops the run if it ever costs more than a dollar.
            </span>
        </td>
    </tr>
</table>

In [7]:
llm = ChatOpenAI(
    model="qwen3.5:4b",
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=120,
    max_retries=5,
)

claim_handler = {
    "name": "claim-handler",
    "description": "Decides one claim and writes its decision file. Give it the claim id.",
    "system_prompt": HANDLER_INSTRUCTIONS,
    "middleware": [ModelRetryMiddleware(max_retries=3)],
}

agent = create_deep_agent(
    model=llm,
    system_prompt=INSTRUCTIONS,
    subagents=[claim_handler],
    middleware=[TodoListMiddleware(), ModelRetryMiddleware(max_retries=3)],
    backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
)

### One adjustment for Windows

The Claude Agent SDK runs Claude Code as a separate program. Inside a Jupyter notebook on Windows, the notebook's event loop can't start other programs, so `query` fails with `CLIConnectionError: Failed to start Claude Code`. The cell below gives `query` an event loop of its own in a background thread, which lets the cells below use it exactly as the SDK documents it. On Mac and Linux it does nothing at all, so run it either way and carry on.

In [8]:
# No Windows adjustment is needed: Deep Agents calls the model over HTTP, not a subprocess.

## Part 3: Working the queue

As the agent works we print each step. The subagent's steps are indented.

In [9]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call["args"]
    if not isinstance(args, dict):
        return tool_call["name"]
    if "file_path" in args or "path" in args:
        return f"{tool_call['name']} {Path(args.get('file_path') or args.get('path')).name}"
    if tool_call["name"] == "task":
        return f"task: {args.get('description', '').splitlines()[0][:80]}"
    if tool_call["name"] == "write_todos":
        return "write_todos: " + ", ".join(
            f"{todo['content'][:30]} ({todo['status']})" for todo in args.get("todos", [])
        )
    return tool_call["name"]

In [10]:
calls = 0
start = time.time()

for namespace, update in agent.stream(
    {"messages": [{"role": "user", "content": "Please work the queue."}]},
    stream_mode="updates",
    subgraphs=True,
    config={"recursion_limit": 200},
):
    if "model" in update:
        indent = "      " if namespace else ""
        for message in update["model"]["messages"]:
            calls += 1
            for tool_call in message.tool_calls:
                print(indent + describe(tool_call))
            if not namespace and message.text.strip():
                print(message.text.strip())

print(f"\nFinished after {calls} model calls in {time.time() - start:.0f} seconds")

ls claims


write_todos: Process claim CLM-1007 (in_progress), Process claim CLM-1015 (pending), Process claim CLM-1017 (pending), Process claim CLM-1020 (pending), Process claim CLM-1021 (pending)
read_file CLM-1007.json
read_file CLM-1015.json
read_file CLM-1017.json
read_file CLM-1020.json
read_file CLM-1021.json


ls policies
read_file policy_wording.md
read_file claims_manual.md


read_file POL-2007.json
read_file POL-2015.json
read_file POL-2017.json
read_file POL-2020.json
read_file POL-2021.json


read_file CUS-3007.json
read_file CUS-3015.json
read_file CUS-3017.json
read_file CUS-3020.json
read_file CUS-3021.json


task: Process claim CLM-1007 about phone dropped in shower with liquid damage. Check i
task: Process claim CLM-1015 about trip interruption due to leaving child at home. Che
task: Process claim CLM-1017 about backpack stolen with missing police report. The cus
task: Process claim CLM-1020 about sunglasses stolen by monkey at Uluwatu Temple. Per 
task: Process claim CLM-1021 about tote bag with electronics stolen in Buenos Aires. P


... (37 outputs left out)

write_file handover.md


ls claims


write_file CLM-1017_decision.json



Finished after 52 model calls in 715 seconds


## Part 4: How it did

Each decision is a file on the claims desk. We compare them with the handlers' decisions, using the same test as on Day 2: the decision has to match and the payout has to be within a dollar.

In [11]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get("decision") == label["decision"] and abs(float(answer.get("payout") or 0) - label["payout"]) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / "decisions" / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
    rows.append({
        "claim": claim_id,
        "decision": answer.get("decision"),
        "payout": answer.get("payout"),
        "handler's decision": label["decision"],
        "handler's payout": label["payout"],
        "right": is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

4 of 5 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,approve,50.0,approve,50.0,True
1,CLM-1015,decline,0.0,decline,0.0,True
2,CLM-1017,refer,0.0,refer,0.0,True
3,CLM-1020,approve,240.0,approve,240.0,True
4,CLM-1021,approve,1053.0,approve,825.0,False


And here's the note it left for the handler.

In [12]:
display(Markdown((DESK / "handover.md").read_text(encoding="utf-8")))

# Claim Processing Summary

All claims have been decided. Below is a summary of each claim's outcome:

## CLM-1007 - Greg Kowalski (Economy plan)
**Decision:** Approve
**Payout:** $50
**Issue:** Phone dropped in shower, liquid damage deemed unreparable by repair shop.
**Notes:** Accidental damage is covered under Section 5. The per-item limit of $150 for Economy plans applies to the electronics category. After subtracting the $100 deductible, payout is $50.

## CLM-1015 - Kate Callister (First plan)
**Decision:** Decline
**Issue:** Trip interrupted due to oversleeping and leaving a child at home.
**Notes:** This reason does not qualify under Section 2 of the policy wording. Covered reasons for trip interruption include sickness/injury/death, severe weather/natural disaster stopping carrier service, home/destination uninhabitable by fire/flood/storm, jury duty, or job loss through no fault of own after one year.

## CLM-1017 - Emily Nguyen (Economy plan)
**Decision:** Refer
**Issue:** Backpack stolen in Naples with missing police report.
**Notes:** Per Section 5 of the policy wording, theft requires a police report made within 24 hours. The customer stated no Italian police report was available. The claim needs to be referred to obtain the required police report before it can be approved or declined.

## CLM-1020 - Nadia Rahman (First plan)
**Decision:** Approve
**Payout:** $240
**Issue:** Sunglasses stolen by monkey at Uluwatu Temple, Bali.
**Notes:** Per the interpretation notes on wild animals in Section 7 of the manual, a loss caused by a wild animal is treated as accidental damage, not theft. No police report is required. A written statement from the tour guide suffices as proof of the incident.

## CLM-1021 - Camille Fontaine (Premium plan)
**Decision:** Approve
**Payout:** $1,053
**Issue:** Electronics stolen from tote bag in Buenos Aires.
**Notes:** Theft was reported to police on the same day as the incident (within 24-hour requirement). The Electronics Plus add-on applies a $1,200 per-item limit for electronics (headphones: $429, e-reader: $180, game console: $399). The tote bag at $95 qualifies without a receipt as it's under the general threshold. Total of $1,103 minus the $50 Premium deductible equals $1,053 payout.

---

## Key Notes for Handler
- CLM-1017 will need follow-up to obtain the police report before further action can be taken on that claim.
- For wild animal theft (CLM-1020), no police report is required per manual interpretation notes.
- Deductibles of $50 (Premium) and $100 (Economy) apply once per claim to Baggage and Personal Effects coverage.
- Electronics Plus add-on extends the per-item limit to $1,200 for covered electronics items.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Build or buy</h2>
            <span style="color:#118AB2;">A ready-made harness like this one gets you a capable agent with very little code. The trade-offs are that it runs only Claude models, and it starts a Claude Code program for every run.<br/><br/>
            Weighing these trade-offs for a customer is a big part of an AI Solutions Architect's job.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Look back through the steps and find where the agent used its task list and where it handed work to the subagent.<br/><br/>
            Then try one change and see what it does to accuracy and cost: a different effort level, a different model such as <code>claude-haiku-4-5</code>, or a change to the instructions.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

## Solution - assignment challenge

**Was this experiment run? No.** No new harness run was made while writing this solution: OpenRouter's free quota is exhausted, and a full harness run is slow and uses many requests. The cells below are ready to run; each variant catches errors, so a failed model does not wipe out earlier results.

### What to change and what to measure

The harness rebuilds an agent for each variant, works the same five claims on a fresh claims desk, and compares the decision files with the handlers' answers using `is_right`. We record:

* **accuracy** — how many of the five claims are right on decision and payout;
* **cost** — from `token_usage["cost"]` when the provider reports it (Ollama is local, so its cost is $0; add your own electricity or GPU rental cost if that matters);
* **time** — wall-clock seconds for the whole run, and the number of model calls.

A single run of five claims is a small sample: one wrong decision moves accuracy by 20 points. Repeat a promising variant a few times before trusting the difference, and beware that the agent can take a different tool path each run.

In [ ]:
# Lab 1 has no OpenRouter connection, so supply a default for the local-only runs.
OPENROUTER_BASE_URL = globals().get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
openrouter_api_key = globals().get("openrouter_api_key")


def run_variant(model_name, effort=None, base_url=None, api_key=None, label=None):
    """Rebuild the harness for one model/effort, work the queue, score it and time it."""
    make_desk(claims)  # fresh desk: old decision files must not leak into this run
    kwargs = {
        "model": model_name,
        "base_url": base_url or OPENROUTER_BASE_URL,
        "api_key": api_key or openrouter_api_key,
        "max_tokens": 4000,  # this account's allowance rejects the 131072 default
        "timeout": 120,
        "max_retries": 5,
    }
    if effort:
        kwargs["reasoning_effort"] = effort
    variant_llm = ChatOpenAI(**kwargs)
    variant_agent = create_deep_agent(
        model=variant_llm,
        system_prompt=INSTRUCTIONS,
        subagents=[claim_handler],
        middleware=[TodoListMiddleware(), ModelRetryMiddleware(max_retries=5, initial_delay=5, max_delay=120)],
        backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
    )

    start = time.time()
    calls = tokens = cost = 0
    for namespace, update in variant_agent.stream(
        {"messages": [{"role": "user", "content": "Please work the queue."}]},
        stream_mode="updates", subgraphs=True, config={"recursion_limit": 200},
    ):
        if "model" in update:
            for message in update["model"]["messages"]:
                calls += 1
                usage = (message.response_metadata or {}).get("token_usage") or {}
                tokens += (usage.get("prompt_tokens") or 0) + (usage.get("completion_tokens") or 0)
                cost += usage.get("cost") or 0.0
    seconds = round(time.time() - start, 1)

    right = 0
    for claim_id, handler in labels.items():
        path = DESK / "decisions" / f"{claim_id}.json"
        answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
        right += is_right(answer, handler)
    return {
        "variant": label or f"{model_name} ({effort or 'default'})",
        "decisions right": f"{right}/{len(labels)}",
        "accuracy (%)": round(100 * right / len(labels)),
        "model calls": calls,
        "tokens": tokens,
        "cost ($)": round(cost, 4),
        "seconds": seconds,
    }


def compare_variants(variants):
    """variants: (model_name, effort, label, base_url, api_key) tuples."""
    rows = []
    for model_name, effort, label, base_url, api_key in variants:
        try:
            rows.append(run_variant(model_name, effort, base_url, api_key, label))
            print(f"Done: {label or model_name}")
        except Exception as error:
            print(f"Not run: {label or model_name}: {type(error).__name__}: {error}")
            rows.append({"variant": label or model_name, "error": f"{type(error).__name__}: {error}"})
    return pd.DataFrame(rows)

### Lab 1 variants

In the SDK version of this lab, effort and model are set in the SDK's run options. This notebook uses Deep Agents on Ollama because no Anthropic key is available, so the equivalent is `reasoning_effort` on `ChatOpenAI`. Ollama's small models may ignore it; changing the model is the more reliable experiment here. `gemma3:1b` may need `ollama pull gemma3:1b` first. The table includes our local baseline so you can see cost as $0.

If you do have an Anthropic key, make the same comparison with the SDK by changing its effort or model options and recording the cost reported by its usage result. Do not mix SDK and Deep Agents costs in one table: they bill different models.

In [ ]:
LAB1_VARIANTS = [
    ("qwen3.5:4b", None, "local baseline", "http://localhost:11434/v1", "ollama"),
    ("qwen3.5:4b", "low", "local, low effort", "http://localhost:11434/v1", "ollama"),
    ("gemma3:1b", None, "small local model", "http://localhost:11434/v1", "ollama"),
]

lab1_comparison = compare_variants(LAB1_VARIANTS)
print(lab1_comparison.to_string(index=False))